In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, PreTrainedTokenizer
from collections import defaultdict, deque
import math
import logging
import json
from typing import Dict, List, Tuple, Optional, Union
from dataclasses import dataclass
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm, trange
import wandb
import os
from pathlib import Path
from collections import deque
from torch.utils.data import Dataset
from typing import List, Tuple, Dict, Any, Optional
from collections import Counter
import random
import string
import re
from tqdm import tqdm
from vllm import LLM, SamplingParams
from config import PRMConfig
import torch.multiprocessing as mp
from datasets import load_dataset
from utils import _sanitize_enhanced, _numeric_equiv_enhanced, _extract_boxed_answer, system_prompt

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"  # Arrange GPU devices starting from 0
os.environ["CUDA_VISIBLE_DEVICES"]= "2"
# os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"  # Jupyter forks → spawn 전환
# mp.set_start_method("spawn", force=True) 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INFO 09-09 13:55:56 [__init__.py:244] Automatically detected platform cuda.


ImportError: cannot import name 'system_prompt' from 'utils' (/home/leena/ccc_eval/rs_prm/data_generation/utils.py)

In [3]:
def _pretty_print(prompts: list[str], results, *, nchars: int = 512, show_all_candidates: bool = True, case: str = "Perturbed"):
    for i, (p, r) in enumerate(zip(prompts, results)):
        print(f"\n=== Prompt #{i} {case} ===")
        outs = r.outputs if show_all_candidates else r.outputs[:1]
        for j, cand in enumerate(outs):
            txt = cand.text.strip().replace("\n", " ")
            print(f"  -> cand[{j}]:", txt)
    print("\n") 


class ContriRewardvLLM:
    ANSWER_PATTERN = re.compile(
        r"""^[\s>#*\-]*          # optional markdown/bullet symbols
            Answer               # word 'Answer'
            \s*[:.\-]\s*         # separator
            (.+?)\s*$            # capture everything after
        """,
        re.IGNORECASE | re.MULTILINE | re.VERBOSE,
    )
    _ANSWER_RE = re.compile(r"####\s*(.+?)\s*$")
    _MASK_PATTERN = re.compile(
        r"""
        (?:
            \b\d+(?:\.\d+)?\b         # integers / decimals
          | \b\d+/\d+\b                 # simple fractions
        )
        """,
        re.VERBOSE,
    )
    
    def __init__(self, config: "PRMConfig", model_name: str = "mistralai/Mathstral-7B-v0.1"):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.llm = LLM(
            model=model_name,
            trust_remote_code=True,
            dtype="bfloat16",
            gpu_memory_utilization=0.9,
            max_model_len=4096,
            quantization="bitsandbytes",
        )
        self.tokenizer = self.llm.get_tokenizer()
        
        self.rollout_params = SamplingParams(
            temperature=0.5,
            top_p=0.9,
            max_tokens=self.config.max_new_tokens,
            n=self.config.num_rollouts,
            repetition_penalty=1.1,
        )
        self.masking_params = SamplingParams(
            temperature=0.5,
            top_p=0.9,
            max_tokens=self.config.max_new_tokens,
            n=self.config.num_rollouts,
            repetition_penalty=1.1,
        )
        print(f"vLLM model loaded: {model_name}")

    def _batched_generate(self, prompts: List[str], params: SamplingParams):
        return self.llm.generate(prompts, params)

    def _extract_answer(self, text: str) -> Optional[str]:
        match = self.ANSWER_PATTERN.search(text)
        if match:
            return _sanitize_enhanced(match.group(1))
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        if lines:
            candidate = lines[-1]
            if re.search(r"\d", candidate):  # contains digit
                return _sanitize_enhanced(candidate)
        for line in reversed(text.splitlines()):
            if line.strip().lower().startswith("answer"):
                return _sanitize_enhanced(line.split("Answer", 1)[-1])
        return None

    def _score_batch(self, outputs, gold_answer: str) -> List[float]:
        """Convert vLLM batched outputs → reward list (fraction of correct roll‑outs)."""
        rewards = []
        for result in outputs:
            correct = sum(
                1 for comp in result.outputs
                if (ans := self._extract_answer(comp.text)) and _numeric_equiv_enhanced(ans, gold_answer)
            )
            rewards.append(correct / float(self.config.num_rollouts))
        return rewards

    def compute_step_rewards_batch(self, question: str, sys_prompt: str, steps: List[str], gold_answer: str) -> List[float]:
        base_prompt = f"{sys_prompt}\n\nProblem: {question}\n"
        prompts = [
            base_prompt + "\n".join(steps[:i + 1]) + "\n" + (f"Step {i + 2}:" if i < len(steps) - 1 else "Answer:")
            for i in range(len(steps))
        ]
        outputs = self._batched_generate(prompts, self.rollout_params)
        _pretty_print(prompts, outputs, show_all_candidates=True, case="Original")
        return self._score_batch(outputs, gold_answer)
        
    def model_masking_batch(self, texts: List[str]) -> List[str]:
        mask_prompts = [
            (
                "In the sentence below, mask any word or expression that seems crucial (such as a variable or a number or a operator etc.) "
                "for solving the math problem by replacing it with '[MASKED]'.\n"
                f"Sentence: \"{t}\"\nRewritten:"
            )
            for t in texts
        ]
        outputs = self._batched_generate(mask_prompts, self.masking_params)
        return [out.outputs[0].text.strip() for out in outputs]

    def perturb_step_rewards_batch(self, question: str, sys_prompt: str, steps: List[str], gold_answer: str, use_llm: bool = True) -> List[float]:
        base_prompt = f"{sys_prompt}\n\nProblem: {question}\n"
        bodies = []
        prefixes = []
        for step in steps:
            m = re.match(r"^[\s>#*\-]*Step\s*\d+\s*[:.\-]\s*", step, flags=re.I)
            prefixes.append(m.group(0) if m else "")
            bodies.append(step[len(prefixes[-1]):])

        if use_llm:
            masked_bodies = self.model_masking_batch(bodies)
            print("Masked Bodies:", masked_bodies, flush=True)
        else:
            masked_bodies = [self._MASK_PATTERN.sub("[MASKED]", b) for b in bodies]
            
        prompts = []
        for i in range(len(steps)):
            masked_step = prefixes[i] + masked_bodies[i]
            staged_steps = steps[:i] + [masked_step]
            label = f"Step {i + 2}:" if i < len(steps) - 1 else "Answer:"
            prompts.append(base_prompt + "\n".join(staged_steps) + "\n" + label)

        outputs = self._batched_generate(prompts, self.rollout_params)
        _pretty_print(prompts, outputs, show_all_candidates=True)
        return self._score_batch(outputs, gold_answer)

    def gsm8k_reward_dataset_vllm(self, *, split: str = "train", start: int = 0, take: int | None):
        ds = load_dataset("openai/gsm8k", "main", split=split)
        ds = ds.select(range(start, start + take)) if take else ds
        # ds = ds.select(range(start, len(ds)))
        # print("Generated dataset size: ", len(ds))

        for sample in tqdm(ds, desc="Building GSM8K contri reward-dataset"):
            q_txt, g_sol = sample["question"], sample["answer"]
            lines, gold_ans = [], None
            
            for ln in g_sol.splitlines():
                ln = ln.strip()
                if not ln:
                    continue
                m = self._ANSWER_RE.match(ln)
                if m:
                    gold_ans = _sanitize_enhanced(m.group(1))
                    break
                lines.append(ln)
            if gold_ans is None:
                raise ValueError("gold answer not found for sample")
            
            steps = [f"Step {i+1}: {t}" for i, t in enumerate(lines)]

            ori = self.compute_step_rewards_batch(q_txt, system_prompt("rollout"), steps, gold_ans)
            ptb = self.perturb_step_rewards_batch(q_txt, system_prompt("rollout"), steps, gold_ans, self.config.use_llm)
            contrib = [round(o - p, 4) for o, p in zip(ori, ptb)]

            entry = {
                "question": q_txt,
                "completion": steps,
                "ori_rewards": ori,
                "ptb_rewards": ptb,
                "contributions": contrib,
                "gold_answer": gold_ans,
            }
            yield entry

    def math_reward_dataset_vllm(self, *, split: str = "train", start: int = 0, take: int | None):
        sent_split = re.compile(r'\.(?!\d)(?=\s|$)')
        ds = load_dataset("HuggingFaceTB/MATH", "all", split=split)
        ds = ds.select(range(start, start + take)) if take else ds
        # ds = ds.select(range(start, len(ds)))
        # print("Generated dataset size: ", len(ds))
        
        for sample in tqdm(ds, desc="Building MATH contri reward-dataset"):
            full_sol = sample["solution"]
            boxed_content = _extract_boxed_answer(full_sol)
            gold_ans = _sanitize_enhanced(boxed_content) if boxed_content else None
            if gold_ans is None:
                lines = [line.strip() for line in full_sol.splitlines() if line.strip()]
                for line in reversed(lines):
                    if re.search(r'[\d\-+*/()=]', line):
                        gold_ans = _sanitize_enhanced(line)
                        break
            
            sol_wo_box = re.sub(r'\\boxed\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', '', full_sol)
            raw_steps = [s.strip() for s in sent_split.split(sol_wo_box) if s.strip()]
            steps = [f"Step {i+1}: {s}" for i, s in enumerate(raw_steps)]

            ori = self.compute_step_rewards_batch(sample["problem"], system_prompt("rollout"), steps, gold_ans)
            ptb = self.perturb_step_rewards_batch(sample["problem"], system_prompt("rollout"), steps, gold_ans, self.config.use_llm)
            contrib = [round(o - p, 4) for o, p in zip(ori, ptb)]

            entry = {
                "question": sample["problem"],
                "completion": steps,
                "ori_rewards": ori,
                "ptb_rewards": ptb,
                "contributions": contrib,
                "gold_answer": gold_ans,
            }
            yield entry


In [7]:
gsm = load_dataset("openai/gsm8k", "main", split="train")
for idx in range(0,3):
    print(gsm[idx]['answer'])
    print()

Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10

In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.
#### 5



In [5]:
math = load_dataset("HuggingFaceTB/MATH", "all", split="train")
for idx in range(0,3):
    print(math[idx]['solution'])
    print()

For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be equal when $x=-2$. Substituting, we get $-2-5=2(-2)-b$, which implies $b=3$. So $a+b=-3+3=\boxed{0}$.

Let $x$ be the number of band members in each row for the original formation, when two are left over.  Then we can write two equations from the given information: $$rx+2=m$$ $$(r-2)(x+1)=m$$ Setting these equal, we find: $$rx+2=(r-2)(x+1)=rx-2x+r-2$$ $$2=-2x+r-2$$ $$4=r-2x$$ We know that the band has less than 100 members.  Based on the first equation, we must have $rx$ less than 98.  We can guess and check some values of $r$ and $x$ in the last equation.  If $r=18$, then $x=7$, and $rx=126$ which is too big.  If $r=16$, then $x=6$, and $rx=96$, which is less than 98.  Checking back in the second formation, we see that $(16-2)(6+1)=14\cdo

# Baselines

In [1]:
import re, random, math
import numpy as np
from typing import List, Optional, Any, Dict, Iterable, Tuple
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
from vllm import LLM, SamplingParams
from config import PRMConfig
from utils import (_sanitize_enhanced, _numeric_equiv_enhanced, _extract_boxed_answer, _split_into_steps_omni, 
                   SELF_REFLECTION_BANK, WRONG_STEP_BANK, IRRELEVANT_BANK)

import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"  # Arrange GPU devices starting from 0
os.environ["CUDA_VISIBLE_DEVICES"]= "2"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INFO 09-09 16:33:36 [__init__.py:244] Automatically detected platform cuda.


In [2]:
class BaselineReward:
    ANSWER_PATTERN = re.compile(
        r"""^[\s>#*\-]*          # optional markdown/bullet symbols
            Answer               # word 'Answer'
            \s*[:.\-]\s*         # separator
            (.+?)\s*$            # capture everything after
        """,
        re.IGNORECASE | re.MULTILINE | re.VERBOSE,
    )
    _ANSWER_RE = re.compile(r"####\s*(.+?)\s*$")

    # ---- Optional separate prover backend (HF or vLLM) helpers ----
    class _SimplePiece:
        def __init__(self, text: str):
            self.text = text
    class _SimpleResult:
        def __init__(self, texts: List[str]):
            self.outputs = [BaselineReward._SimplePiece(t) for t in texts]
    
    def __init__(self, config: "PRMConfig", model_name: str = "mistralai/Mathstral-7B-v0.1", 
            backend: str = "hf",                 # NEW: "hf" (recommended for notebook) or "vllm"
            debug: bool = True,                  # NEW: turn on prints
            dbg_max_prompts: int = 2,            # NEW: limit how many prompts to print
            dbg_max_outputs: int = 2,            # NEW: limit how many rollouts per prompt to print
            dbg_max_chars: int = 400,):
        self.config = config
        self.backend = backend
        self.debug = debug
        self.dbg_max_prompts = dbg_max_prompts
        self.dbg_max_outputs = dbg_max_outputs
        self.dbg_max_chars = dbg_max_chars

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        if backend == "vllm":
            from vllm import LLM, SamplingParams  # ensure available
            self.LLM = LLM
            self.SamplingParams = SamplingParams
            self.llm = self.LLM(
                model=model_name,
                trust_remote_code=True,
                dtype="bfloat16",
                gpu_memory_utilization=0.75,
                max_model_len=4096,
            )
            self.tokenizer = self.llm.get_tokenizer()
        else:
            # HF backend (simple & notebook-friendly)
            self.hf_tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            if self.hf_tok.pad_token is None:
                self.hf_tok.pad_token = self.hf_tok.eos_token
            self.hf_model = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map="auto",
                torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
                trust_remote_code=True
            ).eval()
            self.tokenizer = self.hf_tok

        self._judge_llm: Optional[LLM] = None
        self._judge_tok = None

        # default rollout params (emulate vLLM SamplingParams shape)
        # we’ll carry this object around but map it for HF when needed
        self.rollout_params = type("SamplingParamsShim", (), {})()
        self.rollout_params.temperature = 0.6
        self.rollout_params.top_p = 0.95
        self.rollout_params.top_k = 20
        self.rollout_params.max_tokens = self.config.max_new_tokens
        self.rollout_params.n = self.config.num_rollouts
        self.rollout_params.repetition_penalty = 1.05

        print(f"[INIT] backend={self.backend}, model={model_name}")
    
    # ------------------------ Core prompting helpers ------------------------
    def build_prompt(self, question: str, tokenizer) -> str:
        sys_prompt = (
            "You are Qwen-Math, a meticulous math tutor. "
            "Solve the given math problem step by step. "
            "Use the EXACT format:\n"
            "Step 1: <reasoning>\n\n"
            "Step 2: <reasoning>\n\n"
            "...\n\n"
            "Answer: \\boxed{<final answer>}"
        )
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": f"Problem: {question}"},
        ]
        base = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        return base
    
    def build_prompt_with_prefix(self, question: str, prefix_steps: List[str], tokenizer=None) -> str:
        tok = tokenizer or self.tokenizer
        sys_prompt = (
            "You are a meticulous math tutor. Continue the reasoning from the given partial steps "
            "and finish with 'Answer: \\boxed{...}'. Keep the same 'Step k: ...' format."
        )
        prefix_txt = "\n\n".join(prefix_steps) if prefix_steps else ""
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": f"Problem: {question}"},
        ]
        base = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        if prefix_txt:
            base += f"\n{prefix_txt}\n"
        return base
    
    # ------------------------ Generation (vLLM or HF) ------------------------
    def _batched_generate(self, prompts: List[str], params):
        if self.backend == "vllm":
            return self.llm.generate(prompts, params)  # vLLM object
        # HF path: emulate vLLM result shape
        results = []
        do_sample = (params.n > 1) or (params.temperature is not None and params.temperature > 0)
        for p in prompts:
            outputs = []
            for _ in range(max(1, params.n)):
                enc = self.hf_tok(p, return_tensors="pt").to(self.hf_model.device)
                gen = self.hf_model.generate(
                    **enc,
                    do_sample=do_sample,
                    temperature=params.temperature if params.temperature else 0.0,
                    top_p=params.top_p if params.top_p else 1.0,
                    top_k=params.top_k if params.top_k else 50,
                    max_new_tokens=params.max_tokens if params.max_tokens else 128,
                    pad_token_id=self.hf_tok.pad_token_id,
                    repetition_penalty=getattr(params, "repetition_penalty", 1.0),
                )
                # keep only the generated tail (after prompt)
                tail = gen[0, enc["input_ids"].shape[1]:]
                out_text = self.hf_tok.decode(tail, skip_special_tokens=True)
                outputs.append(out_text)
            results.append(BaselineReward._SimpleResult(outputs))
        return results

    # ------------------------ Debug dump ------------------------
    def _debug_dump(self, *, title: str, prompts: List[str], results, gold_answer: Optional[str] = None):
        if not self.debug:
            return
        print(f"\n[DEBUG] {title}")
        show_p = min(self.dbg_max_prompts, len(prompts))
        for i in range(show_p):
            print("="*80)
            print(f"[Prompt #{i}]")
            print(prompts[i])
            outs = results[i].outputs if results[i] and hasattr(results[i], "outputs") else []
            show_o = min(self.dbg_max_outputs, len(outs))
            for j in range(show_o):
                txt = outs[j].text
                short = (txt[:self.dbg_max_chars] + "...") if len(txt) > self.dbg_max_chars else txt
                print("-"*40)
                print(f"[Output {i}.{j}] {short}")
                ans = self._extract_answer(txt)
                if ans is not None:
                    print(f"  -> parsed answer: {ans}")
                    if gold_answer is not None:
                        ok = _numeric_equiv_enhanced(ans, gold_answer)
                        print(f"  -> correct? {ok}")
    
    def _ensure_prover(self, model_name: str = "Qwen/Qwen2.5-1.5B", backend: str = "vllm"):
        """Lazily init a separate prover (μ) model for Advantage term. backend in {"hf", "vllm"}."""
        if not hasattr(self, "_prover"):
            self._prover = None
            self._prover_tok = None
            self._prover_backend = None
        if self._prover is not None and self._prover_backend == backend and getattr(self, "_prover_name", None) == model_name:
            return
        if backend == "vllm":
            self._prover = LLM(model=model_name, trust_remote_code=True, dtype="bfloat16", gpu_memory_utilization=0.85, max_model_len=4096)
            self._prover_tok = self._prover.get_tokenizer()
        else:  # hf
            self._prover_tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            self._prover = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype="auto", trust_remote_code=True).eval()
        self._prover_backend = backend
        self._prover_name = model_name

    def _prover_generate(self, prompts: List[str], params: SamplingParams):
        """Generate with the prover backend and return vLLM-compatible results list."""
        if self._prover_backend == "vllm":
            return self._prover.generate(prompts, params)
        # HF path: loop per prompt, approximate SamplingParams
        results = []
        do_sample = (params.n > 1) or (params.temperature is not None and params.temperature > 0)
        for p in prompts:
            texts = []
            for _ in range(max(1, params.n)):
                inputs = self._prover_tok(p, return_tensors="pt").to(self._prover.device)
                gen = self._prover.generate(
                    **inputs,
                    do_sample=do_sample,
                    temperature=getattr(params, "temperature", 0.6) or 0.6,
                    top_p=getattr(params, "top_p", 0.95) or 0.95,
                    top_k=getattr(params, "top_k", 20) or 20,
                    max_new_tokens=getattr(params, "max_tokens", 128) or 128,
                    num_return_sequences=1,
                    pad_token_id=self._prover_tok.eos_token_id,
                )
                out = self._prover_tok.decode(gen[0], skip_special_tokens=True)
                # keep only the generated tail after prompt to mimic vLLM behavior is unnecessary for scoring
                texts.append(out)
            results.append(BaselineReward._SimpleResult(texts))
        return results
    
    # ------------------------ Parsing helpers ------------------------
    def _extract_answer(self, text: str) -> Optional[str]:
        m = self.ANSWER_PATTERN.search(text)
        if m:
            return _sanitize_enhanced(m.group(1))
        # PRM-style #### answer
        for line in reversed([ln.strip() for ln in text.splitlines() if ln.strip()]):
            m2 = self._ANSWER_RE.match(line)
            if m2:
                return _sanitize_enhanced(m2.group(1))
        # fallback: last numeric-ish line
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        if lines:
            candidate = lines[-1]
            if re.search(r"[0-9]", candidate):
                return _sanitize_enhanced(candidate)
        # explicit "Answer:" somewhere
        for line in reversed(text.splitlines()):
            if line.strip().lower().startswith("answer"):
                return _sanitize_enhanced(line.split("Answer", 1)[-1])
        return None
    
    def _score_one_prompt_outputs(self, result, gold_answer: str) -> float:
        correct = 0
        total = max(1, len(result.outputs))
        # extra per-output debug
        if self.debug:
            print(f"[DEBUG] scoring {total} outputs vs gold={gold_answer}")
        for idx, comp in enumerate(result.outputs):
            ans = self._extract_answer(comp.text)
            ok = (ans and _numeric_equiv_enhanced(ans, gold_answer))
            if self.debug:
                short = comp.text[:200].replace("\n", " ")
                print(f"  - out#{idx}: parsed={ans}  correct? {bool(ok)}  | text[:120]={short!r}")
            if ok:
                correct += 1
        return correct / float(total)
    
    # ── Baselines Reward Generation Algorithms ────────────────────────────────────────────────
    def compute_qval_step_rewards(self, question: str, steps: List[str], gold_answer: str, tokenizer=None) -> List[float]:
        prompts = []
        for i in range(len(steps)):
            prefix = steps[: i + 1]
            prompts.append(self.build_prompt_with_prefix(question, prefix, tokenizer or self.tokenizer))
        results = self._batched_generate(prompts, self.rollout_params)
        self._debug_dump(title="QVAL rollouts", prompts=prompts, results=results, gold_answer=gold_answer)
        rewards = [self._score_one_prompt_outputs(res, gold_answer) for res in results]
        if self.debug:
            print(f"[DEBUG] QVAL rewards: {rewards}")
        return rewards

    def compute_pav_step_rewards(self, question: str, steps: List[str], gold_answer: str, tokenizer=None, *, 
                                 alpha: float = 1.0, prover_model_name: Optional[str] = "Qwen/Qwen2.5-1.5B", prover_backend: str = "hf") -> List[float]:
        empty_prompt = self.build_prompt_with_prefix(question, [], tokenizer or self.tokenizer)
        policy_prompts = [empty_prompt] + [self.build_prompt_with_prefix(question, steps[: i + 1], tokenizer or self.tokenizer) for i in range(len(steps))]
        policy_results = self._batched_generate(policy_prompts, self.rollout_params)
        self._debug_dump(title="PAV policy Q^π rollouts", prompts=policy_prompts, results=policy_results, gold_answer=gold_answer)
        q_pi = [self._score_one_prompt_outputs(res, gold_answer) for res in policy_results]

        if prover_model_name:  # separate prover for A-term
            self._ensure_prover(prover_model_name, backend=prover_backend)
            # use same textual prompts; if backend is vLLM, its tokenizer is inside prover
            prover_prompts = policy_prompts  # identical prefixes
            prover_results = self._prover_generate(prover_prompts, self.rollout_params)
            self._debug_dump(title="PAV prover A^μ rollouts", prompts=policy_prompts, results=prover_results, gold_answer=gold_answer)
            q_mu = [self._score_one_prompt_outputs(res, gold_answer) for res in prover_results]
        else:
            q_mu = q_pi 

        q_pi_empty, q_mu_empty = q_pi[0], q_mu[0]
        q_pi_per = q_pi[1:]
        q_mu_per = q_mu[1:]

        pav_rewards: List[float] = []
        prev_mu = q_mu_empty
        for qi_pi, qi_mu in zip(q_pi_per, q_mu_per):
            A_mu = qi_mu - prev_mu
            pav = qi_pi + alpha * A_mu
            pav_rewards.append(pav)
            prev_mu = qi_mu
        if self.debug:
            print(f"[DEBUG] PAV rewards: {pav_rewards}")
        return pav_rewards

    def compute_adv_step_rewards(self, question: str, steps: List[str], gold_answer: str, tokenizer=None, *, prover_model_name: Optional[str] = None, prover_backend: str = "hf") -> List[float]:
        empty_prompt = self.build_prompt_with_prefix(question, [], tokenizer or self.tokenizer)
        prompts = [empty_prompt] + [self.build_prompt_with_prefix(question, steps[: i + 1], tokenizer or self.tokenizer) for i in range(len(steps))]
        results = self._batched_generate(prompts, self.rollout_params)
        self._debug_dump(title="ADV Q rollouts", prompts=prompts, results=results, gold_answer=gold_answer)
        q_vals = [self._score_one_prompt_outputs(res, gold_answer) for res in results]
        advs: List[float] = []
        prev = q_vals[0]
        for qi in q_vals[1:]:
            advs.append(qi - prev)
            prev = qi
        if self.debug:
            print(f"[DEBUG] ADV rewards: {advs}")
        return advs
    
    # ---------- LLM-as-a-Judge (Generative PRM) ----------
    def _ensure_judge(self, judge_model: str = "Qwen/Qwen3-8B"):
        if self._judge_llm is None:
            if self.backend == "vllm":
                from vllm import LLM, SamplingParams
                self._judge_llm = LLM(
                    model=judge_model,
                    trust_remote_code=True,
                    dtype="bfloat16",
                    gpu_memory_utilization=0.45,
                    max_model_len=4096,
                )
                self._judge_tok = self._judge_llm.get_tokenizer()
                self._judge_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=128, n=1)
            else:
                self._judge_tok = AutoTokenizer.from_pretrained(judge_model, trust_remote_code=True)
                if self._judge_tok.pad_token is None:
                    self._judge_tok.pad_token = self._judge_tok.eos_token
                self._judge_llm = AutoModelForCausalLM.from_pretrained(
                    judge_model, device_map="auto", torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, trust_remote_code=True
                ).eval()
                # simple shim
                self._judge_params = type("SamplingParamsShim", (), {})()
                self._judge_params.temperature = 0.0
                self._judge_params.top_p = 1.0
                self._judge_params.max_tokens = 50
                self._judge_params.n = 1

    def _judge_prompt(self, question: str, steps_prefix: List[str]) -> str:
        sys = (
            "You are a strict math TA. Given the problem and a partial solution prefix, "
            "rate whether the LAST step in the prefix is logically valid and helpful towards the final solution. "
            "Output ONLY a single line of the form 'SCORE: x' where x is a number in [0,1]."
        )
        prefix = "\n".join(steps_prefix) if steps_prefix else "(no steps yet)"
        messages = [
            {"role": "system", "content": sys},
            {"role": "user", "content": f"Problem:\n{question}\n\nPartial Solution:\n{prefix}\n\nReturn just: SCORE: <float>"},
        ]
        return (self._judge_tok if self.backend == "hf" else self._judge_llm.get_tokenizer()).apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    def _parse_score(self, text: str) -> float:
        m = re.search(r"SCORE\s*:\s*([01]?(?:\.\d+)?)", text, flags=re.I)
        if m:
            try:
                x = float(m.group(1))
                return float(min(1.0, max(0.0, x)))
            except Exception:
                pass
        m = re.search(r"([01](?:\.\d+)?)", text)
        if m:
            try:
                x = float(m.group(1)); return float(min(1.0, max(0.0, x)))
            except Exception:
                pass
        return 0.5
    
    def compute_llm_step_rewards(self, question: str, steps: List[str], gold_answer: str, tokenizer=None, *, judge_model: str = "Qwen/Qwen3-8B") -> List[float]:
        self._ensure_judge(judge_model)
        if self.debug:
            print(f"[DEBUG] Judge model: {judge_model} (backend={self.backend})")
        prompts = [self._judge_prompt(question, steps[: i + 1]) for i in range(len(steps))]
        # generate
        if self.backend == "vllm":
            results = self._judge_llm.generate(prompts, self._judge_params)
            raw_texts = [r.outputs[0].text if r.outputs else "" for r in results]
        else:
            raw_texts = []
            for p in prompts:
                enc = self._judge_tok(p, return_tensors="pt").to(self._judge_llm.device)
                gen = self._judge_llm.generate(
                    **enc, do_sample=False, max_new_tokens=self._judge_params.max_tokens, pad_token_id=self._judge_tok.pad_token_id
                )
                tail = gen[0, enc["input_ids"].shape[1]:]
                raw_texts.append(self._judge_tok.decode(tail, skip_special_tokens=True))
        if self.debug:
            for i, t in enumerate(raw_texts[:self.dbg_max_prompts]):
                print(f"[DEBUG] Judge out#{i}: {t}")
        scores = [self._parse_score(t) for t in raw_texts]
        if self.debug:
            print(f"[DEBUG] LLM-as-judge scores: {scores}")
        return scores
    
    # ---------- Outcome-Only Reward (ORM) ----------
    def compute_orm_rewards(self, question: str, steps: List[str], gold_answer: str, tokenizer=None) -> List[float]:
        joined = "\n\n".join(steps)
        final_ans = _extract_boxed_answer(joined) or None
        if final_ans is None:
            prompt = self.build_prompt_with_prefix(question, steps, tokenizer or self.tokenizer)
            one_params = type("SamplingParamsShim", (), {})()
            one_params.temperature = 0.7; one_params.top_p = 0.8; one_params.top_k = 50
            one_params.max_tokens = 128; one_params.n = 1
            out = self._batched_generate([prompt], one_params)[0]
            gen_text = out.outputs[0].text if out.outputs else ""
            final_ans = self._extract_answer(gen_text)
            if self.debug:
                print("[DEBUG] ORM completion preview:")
                print(prompt[:300] + ("..." if len(prompt) > 300 else ""))
                print(gen_text[:self.dbg_max_chars] + ("..." if len(gen_text) > self.dbg_max_chars else ""))
                print(f"[DEBUG] ORM parsed final_ans={final_ans}")
        is_correct = 1.0 if (final_ans and _numeric_equiv_enhanced(_sanitize_enhanced(final_ans), gold_answer)) else 0.0
        rewards = [0.0] * max(1, len(steps))
        rewards[-1] = is_correct
        if self.debug:
            print(f"[DEBUG] ORM rewards: {rewards}")
        return rewards
    
    # ── Perturbation helpers ────────────────────────────────────────────────
    @staticmethod
    def _is_numeric_list(v: Any, expected_len: int) -> bool:
        if not isinstance(v, list) or len(v) != expected_len:
            return False
        try:
            _ = [float(x) for x in v]
            return True
        except Exception:
            return False

    @staticmethod
    def renumber_steps(steps: List[str], start_idx: int = 0) -> List[str]:
        new_steps: List[str] = []
        for i in range(len(steps)):
            s = steps[i]
            content = s.split(":", 1)[1] if ":" in s else s
            new_steps.append(f"Step {start_idx + i + 1}: {content.lstrip()}")
        return new_steps

    @staticmethod
    def _sample_perturbation_type(perturbation_probs: Optional[Dict[str, float]] = None, rng: Optional[random.Random] = None) -> str:
        R = rng or random
        types = ["wrong_step", "irrelevant", "self_reflection"]
        if perturbation_probs:
            items = [(t, max(0.0, float(perturbation_probs.get(t, 0.0)))) for t in types]
            total = sum(w for _, w in items)
            weights = [w / total for _, w in items] if total > 0 else [2/5, 2/5, 1/5]
        else:
            weights = [2/5, 2/5, 1/5]
        return R.choices(types, weights=weights, k=1)[0]

    def create_perturbed_steps(self, steps: List[str], typ: str, insert_pos: int, rng: Optional[random.Random] = None) -> Tuple[List[str], int]:
        R = rng or random
        assert 0 <= insert_pos <= len(steps)
        new_steps = steps.copy()
        if typ == "wrong_step":
            ins = f"Step {insert_pos + 1}: {random.choice(WRONG_STEP_BANK)}"
        elif typ == "irrelevant":
            ins = f"Step {insert_pos + 1}: {random.choice(IRRELEVANT_BANK)}"
        else:
            ins = f"Step {insert_pos + 1}: {random.choice(SELF_REFLECTION_BANK)}"
        new_steps.insert(insert_pos, ins)
        new_steps = self.renumber_steps(new_steps)
        return new_steps, insert_pos
    
    # One-shot inject+label util used by streamers
    def _inject_then_label(self, *, question: str, steps: List[str], gold_answer: str, base_type: str = "qval", perturbation_probs: Optional[Dict[str, float]] = None, 
                           rng: Optional[random.Random] = None, force_wrong_only: bool = False, tokenizer=None,) -> Dict[str, Any]:
        L = len(steps)
        if L == 0:
            return {"completion": steps, "mi_loo": [], "mi_shapley": [], "mi_margin": [], "incorrect_mask": []}

        R = rng or random
        insert_pos = R.randint(0, L)
        ptype = "wrong_step" if force_wrong_only else self._sample_perturbation_type(perturbation_probs, rng=R)
        new_steps, ins_idx = self.create_perturbed_steps(steps, ptype, insert_pos, rng=R)

        if self.debug:
            print("\n[DEBUG] === Inject & Label ===")
            print(f"  - perturbation: {ptype} at index={ins_idx}")
            print("  - injected steps preview:")
            print("\n".join(new_steps[:min(len(new_steps), 6)]))
            if len(new_steps) > 6: print("  ...")

        # Compute baseline rewards on the *injected* sequence
        if base_type == "qval":
            base_reward = self.compute_qval_step_rewards(question, new_steps, gold_answer, tokenizer=tokenizer)
        elif base_type == "pav":
            base_reward = self.compute_pav_step_rewards(question, new_steps, gold_answer, tokenizer=tokenizer)
        elif base_type == "adv": # policy-only advantage by default
            base_reward = self.compute_adv_step_rewards(question, new_steps, gold_answer, tokenizer=tokenizer)
        elif base_type == "llm":
            base_reward = self.compute_llm_step_rewards(question, new_steps, gold_answer, tokenizer=tokenizer)
        elif base_type == "orm":
            base_reward = self.compute_orm_rewards(question, new_steps, gold_answer, tokenizer=tokenizer)
        else:
            raise ValueError("No valid baseline reward type.")

        # Incorrect mask
        incorrect_mask = [0] * len(new_steps)
        incorrect_mask[ins_idx] = 1

        if self.debug:
            print(f"[DEBUG] base_type={base_type} rewards: {base_reward}")
            print(f"[DEBUG] incorrect_mask: {incorrect_mask}")

        return {
            "completion": new_steps,
            "base_reward": base_reward,
            "base_type": base_type,
            "incorrect_mask": incorrect_mask,
            "perturbation": ptype,
            "perturbation_pos": ins_idx,
        }
    
    # ── Dataset generation ────────────────────────────────────────────────
    def gsm8k_reward_dataset_vllm(self, *, split: str = "train", start: int = 0, take: int | None = 3, base_type: str = "qval",
                                  perturbation_probs: Optional[Dict[str, float]] = None, force_wrong_only: bool = False, rng: Optional[random.Random] = None,):
        ds = load_dataset("openai/gsm8k", "main", split=split)
        ds = ds.select(range(start, start + take)) if take else ds
        print("Generated dataset size: ", len(ds))

        for idx, sample in enumerate(tqdm(ds, desc="Building GSM8K contri reward-dataset")):
            q_txt, g_sol = sample["question"], sample["answer"]
            lines, gold_ans = [], None

            if self.debug:
                print("\n" + "#"*90)
                print(f"[SAMPLE {idx}]")
                print(f"[Question]\n{q_txt}")
            
            for ln in g_sol.splitlines():
                ln = ln.strip()
                if not ln:
                    continue
                m = self._ANSWER_RE.match(ln)
                if m:
                    gold_ans = _sanitize_enhanced(m.group(1))
                    break
                lines.append(ln)
            if gold_ans is None:
                raise ValueError("gold answer not found for sample")
            
            if self.debug:
                print(f"[Gold Answer] {gold_ans}")
            
            steps = [f"Step {i+1}: {t}" for i, t in enumerate(lines)]

            if self.debug:
                print("[Gold Steps Preview]")
                print("\n".join(steps[:min(5, len(steps))]))
                if len(steps) > 5: print("...")

            labeled = self._inject_then_label(question=q_txt, steps=steps, gold_answer=gold_ans, base_type=base_type,
                perturbation_probs=perturbation_probs, rng=rng, force_wrong_only=force_wrong_only, tokenizer=self.tokenizer)

            entry = {
                "question": q_txt,
                "completion": steps,
                "gold_answer": gold_ans,
                "base_reward": labeled["base_reward"],
                "base_type": labeled['base_type'],
                "incorrect_mask": labeled["incorrect_mask"],
                "perturbation": labeled["perturbation"],
                "perturbation_pos": labeled["perturbation_pos"],
            }

            if self.debug:
                print("[OUTPUT] base_reward=", entry["base_reward"])
                print("[OUTPUT] incorrect_mask=", entry["incorrect_mask"])
                print("[OUTPUT] perturbation=", entry["perturbation"], "at", entry["perturbation_pos"])

            yield entry

    def math_reward_dataset_streaming(self, *, split: str = "train", start: int = 0, take: Optional[int] = 0, base_type: str = "qval",
        perturbation_probs: Optional[Dict[str, float]] = None, force_wrong_only: bool = False, rng: Optional[random.Random] = None,) -> Iterable[Dict]:

        sent_split = re.compile(r'\.(?!\d)(?=\s|$)')
        ds = load_dataset("HuggingFaceTB/MATH", "all", split=split)
        ds = ds.select(range(start, start + take)) if take else ds
        print("Dataset", len(ds), "Loading!")

        for idx, sample in enumerate(tqdm(ds, desc="Building MATH MI reward-dataset")):
            full_sol   = sample["solution"]
            q_txt = sample["problem"]

            if self.debug:
                print("\n" + "#"*90)
                print(f"[SAMPLE {idx}]")
                print(f"[Question]\n{q_txt}")

            boxed_content = _extract_boxed_answer(full_sol)
            gold_ans = _sanitize_enhanced(boxed_content) if boxed_content else None
            if gold_ans is None:
                lines = [line.strip() for line in full_sol.splitlines() if line.strip()]
                for line in reversed(lines):
                    if re.search(r'[\d\-+*/()=]', line):
                        gold_ans = _sanitize_enhanced(line)
                        break
            
            if self.debug:
                print(f"[Gold Answer] {gold_ans}")
            
            raw_steps = [s.strip() for s in sent_split.split(full_sol) if s.strip()]
            steps = [f"Step {i+1}: {s}" for i, s in enumerate(raw_steps)]

            if self.debug:
                print("[Gold Steps Preview]")
                print("\n".join(steps[:min(5, len(steps))]))
                if len(steps) > 5: print("...")

            labeled = self._inject_then_label(question=q_txt, steps=steps, gold_answer=gold_ans, base_type=base_type, 
                perturbation_probs=perturbation_probs, rng=rng, force_wrong_only=force_wrong_only, tokenizer=self.tokenizer)
            
            entry = {
                "question": q_txt,
                "completion": labeled["completion"],
                "gold_answer": gold_ans,
                "base_reward": labeled["base_reward"],
                "base_type": labeled['base_type'],
                "incorrect_mask": labeled["incorrect_mask"],
                "perturbation": labeled["perturbation"],
                "perturbation_pos": labeled["perturbation_pos"],
            }

            if self.debug:
                print("[OUTPUT] base_reward=", entry["base_reward"])
                print("[OUTPUT] incorrect_mask=", entry["incorrect_mask"])
                print("[OUTPUT] perturbation=", entry["perturbation"], "at", entry["perturbation_pos"])

            yield entry


In [3]:
class PRMConfig:
    max_new_tokens = 256
    num_rollouts = 4

cfg = PRMConfig()
rewarder = BaselineReward(
    cfg,
    model_name="Qwen/Qwen3-4B-base",  # 가벼운 모델 권장
    backend="hf",                             # 노트북에서는 HF가 편함
    debug=True,
    dbg_max_prompts=4,
    dbg_max_outputs=4,
    dbg_max_chars=600,
)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

[INIT] backend=hf, model=Qwen/Qwen3-4B-base


## qval

In [4]:
# 실제 실행 (take=3)
gen = rewarder.math_reward_dataset_streaming(split="train", start=500, take=3, base_type="qval")
_ = list(gen)  # 출력만 확인

Dataset 3 Loading!


Building MATH MI reward-dataset:   0%|          | 0/3 [00:00<?, ?it/s]


##########################################################################################
[SAMPLE 0]
[Question]
A $100$-gon $P_1$ is drawn in the Cartesian plane.  The sum of the $x$-coordinates of the $100$ vertices equals 2009.  The midpoints of the sides of $P_1$ form a second $100$-gon, $P_2$.  Finally, the midpoints of the sides of $P_2$ form a third $100$-gon, $P_3$.  Find the sum of the $x$-coordinates of the vertices of $P_3$.
[Gold Answer] 2009
[Gold Steps Preview]
Step 1: Let the $x$-coordinates of the vertices of $P_1$ be $x_1,x_2,\ldots,x_{100}$
Step 2: Then, by the midpoint formula, the $x$-coordinates of the vertices of $P_2$ are $\frac{x_1+x_2}2,\frac{x_2+x_3}2,\ldots,\frac{x_{100}+x_1}2 $
Step 3: The sum of these equals $\frac{2x_1+2x_2+\cdots +2x_{100}}2=x_1+x_2+\cdots+x_{100}$
Step 4: Similarly, the sum of the $x$-coordinates of the vertices of $P_3$ equals the sum of the $x$-coordinates of the vertices of  $P_2$
Step 5: Thus the desired answer is $\boxed{2009}$

[D

Building MATH MI reward-dataset:  33%|███▎      | 1/3 [02:08<04:16, 128.10s/it]


[DEBUG] QVAL rollouts
[Prompt #0]
<|im_start|>system
You are a meticulous math tutor. Continue the reasoning from the given partial steps and finish with 'Answer: \boxed{...}'. Keep the same 'Step k: ...' format.<|im_end|>
<|im_start|>user
Problem: A $100$-gon $P_1$ is drawn in the Cartesian plane.  The sum of the $x$-coordinates of the $100$ vertices equals 2009.  The midpoints of the sides of $P_1$ form a second $100$-gon, $P_2$.  Finally, the midpoints of the sides of $P_2$ form a third $100$-gon, $P_3$.  Find the sum of the $x$-coordinates of the vertices of $P_3$.<|im_end|>
<|im_start|>assistant

Step 1: Let the $x$-coordinates of the vertices of $P_1$ be $x_1,x_2,\ldots,x_{100}$

----------------------------------------
[Output 0.0] Step 2: The sum of the $x$-coordinates of the vertices of $P_1$ is $\sum_{i=1}^{100} x_i = 2009$
Step 3: The $x$-coordinates of the vertices of $P_2$ are the midpoints of the sides of $P_1$, so they are $\frac{x_1+x_2}{2},\frac{x_2+x_3}{2},\ldots,\fr

Building MATH MI reward-dataset:  67%|██████▋   | 2/3 [04:24<02:13, 133.24s/it]


[DEBUG] QVAL rollouts
[Prompt #0]
<|im_start|>system
You are a meticulous math tutor. Continue the reasoning from the given partial steps and finish with 'Answer: \boxed{...}'. Keep the same 'Step k: ...' format.<|im_end|>
<|im_start|>user
Problem: Suppose the point $(1,2)$ is on the graph of $y=\frac{f(x)}2$. Then there is one point which must be on the graph of $y=\frac{f^{-1}(x)}{2}$. What is the sum of that point's coordinates?<|im_end|>
<|im_start|>assistant

Step 1: Since $(1,2)$ is on the graph of $y=\frac{f(x)}2$, we know that $$2 = \frac{f(1)}{2},$$which implies that $f(1)=4$

----------------------------------------
[Output 0.0] Step 2: Now we want to find the inverse function $f^{-1}$ and its value at some input.
Step 3: Let $y=f^{-1}(x)$. Then by definition, $f(y)=x$.
Step 4: We already know that $f(1)=4$, so setting $x=4$ in the equation $f(y)=x$, we get $f(y)=4$, which means $y=1$ because we chose $y$ to be the output of $f^{-1}(x)$.
Step 5: Therefore, we have found that

Building MATH MI reward-dataset: 100%|██████████| 3/3 [04:54<00:00, 98.25s/it] 


[DEBUG] QVAL rollouts
[Prompt #0]
<|im_start|>system
You are a meticulous math tutor. Continue the reasoning from the given partial steps and finish with 'Answer: \boxed{...}'. Keep the same 'Step k: ...' format.<|im_end|>
<|im_start|>user
Problem: What is the value of $\displaystyle\frac{235^2-221^2}{14}$?<|im_end|>
<|im_start|>assistant

Step 1: The key to this problem is noticing that $235^2 - 221^2$ factors into $(235+221)(235-221)$

----------------------------------------
[Output 0.0] Step 2: Therefore, we can rewrite the expression as $\frac{(235+221)(235-221)}{14}$
Step 3: Simplifying the numerator gives us $\frac{456 \cdot 14}{14}$
Step 4: The $14$ in the numerator and denominator cancel out, leaving us with $456$
Answer: \boxed{456}

:convert
:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:convert:con

## pav

In [5]:
gen2 = rewarder.gsm8k_reward_dataset_vllm(split="train", start=600, take=2, base_type="pav")
_ = list(gen2)  # 출력만 확인

Generated dataset size:  2


Building GSM8K contri reward-dataset:   0%|          | 0/2 [00:00<?, ?it/s]


##########################################################################################
[SAMPLE 0]
[Question]
Mike can type 65 words per minute. Due to a minor accident, Mike cannot use his right hand for a while so that his typing speed is now 20 words less per minute. If he is supposed to type a document with 810 words, how many minutes will it take him to finish typing the document?
[Gold Answer] 18
[Gold Steps Preview]
Step 1: After the accident, Mike can only type 65 words/minute - 20 words/minute = <<65-20=45>>45 words/minute.
Step 2: So, he will be able to finish typing the document in 810 words / 45 words/minute = <<810/45=18>>18 minutes.

[DEBUG] === Inject & Label ===
  - perturbation: wrong_step at index=2
  - injected steps preview:
Step 1: After the accident, Mike can only type 65 words/minute - 20 words/minute = <<65-20=45>>45 words/minute.
Step 2: So, he will be able to finish typing the document in 810 words / 45 words/minute = <<810/45=18>>18 minutes.
Step 3: Pytha

Building GSM8K contri reward-dataset:  50%|█████     | 1/2 [01:05<01:05, 65.01s/it]


[DEBUG] PAV policy Q^π rollouts
[Prompt #0]
<|im_start|>system
You are a meticulous math tutor. Continue the reasoning from the given partial steps and finish with 'Answer: \boxed{...}'. Keep the same 'Step k: ...' format.<|im_end|>
<|im_start|>user
Problem: Mike can type 65 words per minute. Due to a minor accident, Mike cannot use his right hand for a while so that his typing speed is now 20 words less per minute. If he is supposed to type a document with 810 words, how many minutes will it take him to finish typing the document?<|im_end|>
<|im_start|>assistant

----------------------------------------
[Output 0.0] Step 1: Find Mike's new typing speed after the accident.
Step 2: Calculate the time it takes for Mike to type the entire document.
Step 3: Convert the time from seconds to minutes.
Step 4: Determine the number of minutes it will take Mike to finish typing the document.
Steps:
Step 1: Find Mike's new typing speed after the accident.
Mike's original typing speed is 65 words

Building GSM8K contri reward-dataset: 100%|██████████| 2/2 [02:57<00:00, 88.84s/it]


[DEBUG] PAV policy Q^π rollouts
[Prompt #0]
<|im_start|>system
You are a meticulous math tutor. Continue the reasoning from the given partial steps and finish with 'Answer: \boxed{...}'. Keep the same 'Step k: ...' format.<|im_end|>
<|im_start|>user
Problem: A cheetah can run at a top speed of 60 mph.  The gazelle can run for speeds of up to 40 miles per hour.  If one mile per hour is about 1.5 feet per second, then how many seconds would it take for a cheetah traveling at top speed to catch up to a fleeing gazelle also running at top speed if the two animals were initially 210 feet apart and they both traveled in the same direction?<|im_end|>
<|im_start|>assistant

----------------------------------------
[Output 0.0] Step 1: Convert the top speeds of the cheetah and the gazelle from miles per hour (mph) to feet per second (ft/s).
Step 2: Calculate the relative speed between the cheetah and the gazelle.
Step 3: Determine the time it takes for the cheetah to catch up to the gazelle us

## adv

In [7]:
gen3 = rewarder.gsm8k_reward_dataset_vllm(split="train", start=600, take=2, base_type="adv")
_ = list(gen3)  # 출력만 확인

Generated dataset size:  2


Building GSM8K contri reward-dataset:   0%|          | 0/2 [00:00<?, ?it/s]


##########################################################################################
[SAMPLE 0]
[Question]
Mike can type 65 words per minute. Due to a minor accident, Mike cannot use his right hand for a while so that his typing speed is now 20 words less per minute. If he is supposed to type a document with 810 words, how many minutes will it take him to finish typing the document?
[Gold Answer] 18
[Gold Steps Preview]
Step 1: After the accident, Mike can only type 65 words/minute - 20 words/minute = <<65-20=45>>45 words/minute.
Step 2: So, he will be able to finish typing the document in 810 words / 45 words/minute = <<810/45=18>>18 minutes.

[DEBUG] === Inject & Label ===
  - perturbation: irrelevant at index=0
  - injected steps preview:
Step 1: I should make a grocery list.
Step 2: After the accident, Mike can only type 65 words/minute - 20 words/minute = <<65-20=45>>45 words/minute.
Step 3: So, he will be able to finish typing the document in 810 words / 45 words/minute = 

Building GSM8K contri reward-dataset:  50%|█████     | 1/2 [01:09<01:09, 69.50s/it]


[DEBUG] ADV Q rollouts
[Prompt #0]
<|im_start|>system
You are a meticulous math tutor. Continue the reasoning from the given partial steps and finish with 'Answer: \boxed{...}'. Keep the same 'Step k: ...' format.<|im_end|>
<|im_start|>user
Problem: Mike can type 65 words per minute. Due to a minor accident, Mike cannot use his right hand for a while so that his typing speed is now 20 words less per minute. If he is supposed to type a document with 810 words, how many minutes will it take him to finish typing the document?<|im_end|>
<|im_start|>assistant

----------------------------------------
[Output 0.0] Step 1: Determine Mike's new typing speed after the accident.
Step 2: Calculate the time required for Mike to type the entire document at his reduced typing speed.
Step 1: Mike's original typing speed is 65 words per minute. After the accident, his typing speed is reduced by 20 words per minute.
Step 2: Mike's new typing speed = 65 - 20 = 45 words per minute.
Step 3: The total num

Building GSM8K contri reward-dataset: 100%|██████████| 2/2 [02:49<00:00, 84.75s/it]


[DEBUG] ADV Q rollouts
[Prompt #0]
<|im_start|>system
You are a meticulous math tutor. Continue the reasoning from the given partial steps and finish with 'Answer: \boxed{...}'. Keep the same 'Step k: ...' format.<|im_end|>
<|im_start|>user
Problem: A cheetah can run at a top speed of 60 mph.  The gazelle can run for speeds of up to 40 miles per hour.  If one mile per hour is about 1.5 feet per second, then how many seconds would it take for a cheetah traveling at top speed to catch up to a fleeing gazelle also running at top speed if the two animals were initially 210 feet apart and they both traveled in the same direction?<|im_end|>
<|im_start|>assistant

----------------------------------------
[Output 0.0] Step 1: Convert the top speeds of the cheetah and the gazelle from miles per hour to feet per second.
Step 2: Calculate the relative speed between the cheetah and the gazelle.
Step 3: Determine the time it takes for the cheetah to catch up to the gazelle based on their initial d

## llm

In [4]:
# gen4 = rewarder.gsm8k_reward_dataset_vllm(split="train", start=600, take=2, base_type="llm")
gen4 = rewarder.math_reward_dataset_streaming(split="train", start=600, take=2, base_type="llm")
_ = list(gen4)  # 출력만 확인

Dataset 2 Loading!


Building MATH MI reward-dataset:   0%|          | 0/2 [00:00<?, ?it/s]


##########################################################################################
[SAMPLE 0]
[Question]
If the system of equations

\begin{align*}
6x-4y&=a,\\
6y-9x &=b.
\end{align*}has a solution $(x, y)$ where $x$ and $y$ are both nonzero, find $\frac{a}{b},$ assuming $b$ is nonzero.
[Gold Answer] -(2)/(3)
[Gold Steps Preview]
Step 1: If we multiply the first equation by $-\frac{3}{2}$, we obtain

$$6y-9x=-\frac{3}{2}a.$$Since we also know that $6y-9x=b$, we have

$$-\frac{3}{2}a=b\Rightarrow\frac{a}{b}=\boxed{-\frac{2}{3}}.$$

[DEBUG] === Inject & Label ===
  - perturbation: wrong_step at index=0
  - injected steps preview:
Step 1: If x/y = 2/3, then x = y (cross-multiplication error)
Step 2: If we multiply the first equation by $-\frac{3}{2}$, we obtain

$$6y-9x=-\frac{3}{2}a.$$Since we also know that $6y-9x=b$, we have

$$-\frac{3}{2}a=b\Rightarrow\frac{a}{b}=\boxed{-\frac{2}{3}}.$$


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Building MATH MI reward-dataset:  50%|█████     | 1/2 [00:10<00:10, 10.76s/it]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[DEBUG] Judge out#0: <think>
Okay, let's see. The problem is about a system of equations and finding the ratio a/b. The partial solution starts with a step that says if x/y = 2/3, then x = y, which is a cross
[DEBUG] Judge out#1: <think>
Okay, let's see. The problem is about a system of equations and finding the ratio a/b. The partial solution provided has two steps. Let me check each step for validity.

First, the problem states that the system has a solution
[DEBUG] LLM-as-judge scores: [0.5, 0.5]
[DEBUG] base_type=llm rewards: [0.5, 0.5]
[DEBUG] incorrect_mask: [1, 0]
[OUTPUT] base_reward= [0.5, 0.5]
[OUTPUT] incorrect_mask= [1, 0]
[OUTPUT] perturbation= wrong_step at 0

##########################################################################################
[SAMPLE 1]
[Question]
Let $f(x) = 3x-8$ and $g(f(x)) = 2x^2 + 5x - 3.$ Find $g(-5).$
[Gold Answer] 4
[Gold Steps Preview]
Step 1: We don't know $g(x),$ so we don't have an expression we can simply stick $-5$ in to get an answe

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Building MATH MI reward-dataset: 100%|██████████| 2/2 [00:16<00:00,  8.41s/it]

[DEBUG] Judge out#0: <think>
Okay, let's see. The problem is to find g(-5) given that f(x) = 3x - 8 and g(f(x)) = 2x² + 5x - 3. 

The partial
[DEBUG] Judge out#1: <think>
Okay, let's see. The problem is to find g(-5) given that f(x) = 3x - 8 and g(f(x)) = 2x² + 5x - 3. The partial solution
[DEBUG] Judge out#2: <think>
Okay, let's see. The problem is to find g(-5) given that f(x) = 3x - 8 and g(f(x)) = 2x² +5x -3. The partial solution starts by
[DEBUG] LLM-as-judge scores: [0.5, 0.5, 0.5]
[DEBUG] base_type=llm rewards: [0.5, 0.5, 0.5]
[DEBUG] incorrect_mask: [0, 1, 0]
[OUTPUT] base_reward= [0.5, 0.5, 0.5]
[OUTPUT] incorrect_mask= [0, 1, 0]
[OUTPUT] perturbation= wrong_step at 1


: 

## orm

In [8]:
gen5 = rewarder.gsm8k_reward_dataset_vllm(split="train", start=600, take=2, base_type="orm")
_ = list(gen5)  # 출력만 확인

Generated dataset size:  2


Building GSM8K contri reward-dataset:   0%|          | 0/2 [00:00<?, ?it/s]


##########################################################################################
[SAMPLE 0]
[Question]
Mike can type 65 words per minute. Due to a minor accident, Mike cannot use his right hand for a while so that his typing speed is now 20 words less per minute. If he is supposed to type a document with 810 words, how many minutes will it take him to finish typing the document?
[Gold Answer] 18
[Gold Steps Preview]
Step 1: After the accident, Mike can only type 65 words/minute - 20 words/minute = <<65-20=45>>45 words/minute.
Step 2: So, he will be able to finish typing the document in 810 words / 45 words/minute = <<810/45=18>>18 minutes.

[DEBUG] === Inject & Label ===
  - perturbation: wrong_step at index=0
  - injected steps preview:
Step 1: Circumference of a circle with r=3 is 3r
Step 2: After the accident, Mike can only type 65 words/minute - 20 words/minute = <<65-20=45>>45 words/minute.
Step 3: So, he will be able to finish typing the document in 810 words / 45 word

Building GSM8K contri reward-dataset:  50%|█████     | 1/2 [00:00<00:00,  1.96it/s]

[DEBUG] ORM completion preview:
<|im_start|>system
You are a meticulous math tutor. Continue the reasoning from the given partial steps and finish with 'Answer: \boxed{...}'. Keep the same 'Step k: ...' format.<|im_end|>
<|im_start|>user
Problem: Mike can type 65 words per minute. Due to a minor accident, Mike cannot use his right...
Answer: \boxed{18}
[DEBUG] ORM parsed final_ans=18
[DEBUG] ORM rewards: [0.0, 0.0, 1.0]
[DEBUG] base_type=orm rewards: [0.0, 0.0, 1.0]
[DEBUG] incorrect_mask: [1, 0, 0]
[OUTPUT] base_reward= [0.0, 0.0, 1.0]
[OUTPUT] incorrect_mask= [1, 0, 0]
[OUTPUT] perturbation= wrong_step at 0

##########################################################################################
[SAMPLE 1]
[Question]
A cheetah can run at a top speed of 60 mph.  The gazelle can run for speeds of up to 40 miles per hour.  If one mile per hour is about 1.5 feet per second, then how many seconds would it take for a cheetah traveling at top speed to catch up to a fleeing gazelle also ru

Building GSM8K contri reward-dataset: 100%|██████████| 2/2 [00:00<00:00,  2.19it/s]

[DEBUG] ORM completion preview:
<|im_start|>system
You are a meticulous math tutor. Continue the reasoning from the given partial steps and finish with 'Answer: \boxed{...}'. Keep the same 'Step k: ...' format.<|im_end|>
<|im_start|>user
Problem: A cheetah can run at a top speed of 60 mph.  The gazelle can run for speeds of up to ...
Answer: \boxed{7}
[DEBUG] ORM parsed final_ans=7
[DEBUG] ORM rewards: [0.0, 0.0, 0.0, 0.0, 1.0]
[DEBUG] base_type=orm rewards: [0.0, 0.0, 0.0, 0.0, 1.0]
[DEBUG] incorrect_mask: [0, 0, 0, 1, 0]
[OUTPUT] base_reward= [0.0, 0.0, 0.0, 0.0, 1.0]
[OUTPUT] incorrect_mask= [0, 0, 0, 1, 0]
[OUTPUT] perturbation= self_reflection at 3
